# CIL Monocular Depth - SAM Preprocessing

**Before running:** Runtime -> Change runtime type -> A100 / G4.

#### SAM-v2 Pipeline

For each RGB image in the dataset:

1. `SamAutomaticMaskGenerator` proposes candidate segments as in the offical SAM repo ("git+https://github.com/facebookresearch/segment-anything.git")
2. Each mask is scored via the `predicted_iou * stability_score` outputs of the SAM model, masks below `mask_score_min` are discarded.
3. The remaining masks are sorted by score and rasterized based on the `points_per_side` parameter into per-pixel top-1 and top-2 assignments: `label_top1`, `label_top2`, plus their raw scores.
4. The raw scores are normalized into `conf_top1` and `conf_top2`, and pixel uncertainty is computed as `1 - |p1 - p2|`.
5. `sam_boundary` is derived from discontinuities in `label_top1`, while `boundary_uncertainty` is derived from neighboring top-1 confidences.
6. The notebook writes SAM features for all training images into `train_sam_v2` and all test images into `test_sam_v2`.
7. `src/train_refiner.py` then creates its own train and validation split from the full training set, so no separate SAM-side val split is needed.

In [ ]:
# config
REPO_URL = 'git@github.com:xSurus/CIL_Monocular_depth.git'
BRANCH = 'official-submission-repo'
KAGGLE_TOKEN = '...'  # enter your kaggle token

SAM_RUN_NAME = 'sam_prep_v2'  # export folder name
SAM_MODEL_TYPE = 'vit_h'  # sam backbone
SAVE_TO_DRIVE = True  # copy final bundle to drive
FORCE_REGENERATE_SAM = False  # ignore existing npz files
FORCE_CLEAN_LOCAL_SAM = False  # wipe local sam cache first

MAX_TRAINVAL_IMAGES = None  # optional debug cap for train images
MAX_TEST_IMAGES = None  # optional debug cap for test images
SEED = 42  # used only for optional subset sampling
SUBSET_POLICY = 'random'  # random or first subset

POINTS_PER_SIDE = 24  # mask density
PRED_IOU_THRESH = 0.86
STABILITY_THRESH = 0.92
MASK_SCORE_MIN = 0.0
PART1_STATS_CSV = None  # (optional) extra metadata csv

DRIVE_BASE = '/gdrive/MyDrive/CIL'
DRIVE_DATA_DIR = f'{DRIVE_BASE}/data'
DRIVE_SAM_EXPORT = f'{DRIVE_BASE}/sam_v2/{SAM_RUN_NAME}'  # drive export dir
DRIVE_SAM_CHECKPOINTS = f'{DRIVE_BASE}/sam_checkpoints'
SSH_KEY_PATH = f'{DRIVE_BASE}/id_ed25519'
TRAIN_SAM_ZIP = f'{DRIVE_DATA_DIR}/train_sam_v2.zip'  # all training sam zip for downstream
SAM_ZIP = f'{DRIVE_DATA_DIR}/test_sam_v2.zip'  # test sam zip for downstream

DATA_LOCAL = '/content/data'
DATASET_LOCAL = f'{DATA_LOCAL}/monodepth_kaggle2026'
LOCAL_SAM_ROOT = f'{DATA_LOCAL}/sam'
LOCAL_TRAIN_SAM_DIR = f'{LOCAL_SAM_ROOT}/train_sam_v2'
LOCAL_TEST_SAM_DIR = f'{LOCAL_SAM_ROOT}/test_sam_v2'

#### Setup

In [ ]:
# imports
import os
import sys
import json
import shutil
import importlib
from pathlib import Path
from IPython.display import display
from google.colab import drive

In [ ]:
# mount google drive 
drive.mount('/gdrive')

In [ ]:
# ssh key
if not os.path.exists(SSH_KEY_PATH):
    !mkdir -p /root/.ssh {DRIVE_BASE}
    !ssh-keygen -t ed25519 -C 'colab' -f /root/.ssh/id_ed25519 -N '' -q
    !cp /root/.ssh/id_ed25519 {SSH_KEY_PATH}
    !cp /root/.ssh/id_ed25519.pub {SSH_KEY_PATH}.pub
    print('*** Add this key to github.com/settings/keys, then re-run this cell ***')
    !cat /root/.ssh/id_ed25519.pub
else:
    !mkdir -p /root/.ssh
    !cp {SSH_KEY_PATH} /root/.ssh/id_ed25519
    !cp {SSH_KEY_PATH}.pub /root/.ssh/id_ed25519.pub
    !chmod 600 /root/.ssh/id_ed25519
    !ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null
    print('SSH key loaded from Drive.')

In [ ]:
# sync repo and reload helper if needed
if not os.path.exists('/content/CIL_Monocular_depth'):
    !git clone -b {BRANCH} {REPO_URL} /content/CIL_Monocular_depth
else:
    !git -C /content/CIL_Monocular_depth remote set-url origin {REPO_URL}
    !git -C /content/CIL_Monocular_depth fetch origin
    !git -C /content/CIL_Monocular_depth reset --hard origin/{BRANCH}

%cd /content/CIL_Monocular_depth
sys.path.insert(0, '/content/CIL_Monocular_depth')
importlib.invalidate_caches()
if 'tools.colab_sam_preprocessing' in sys.modules:
    import tools.colab_sam_preprocessing as _sam_utils
    importlib.reload(_sam_utils)

print('Repo ready.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Disk usage for /content before copy/extract:
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G  120G  116G  51% /
Local ZIP already present; skipping copy.
Extract directory already populated; skipping extraction.
DATA_ROOT: /content/data/monodepth_kaggle2026
Top contents: ['create_submission.py', 'test', 'train']
Disk usage for /content after extract:
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G  120G  116G  51% /


In [ ]:
# fetch data prepare folders and build the sam config
if not os.path.exists(DATASET_LOCAL):
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
    !pip install -q kaggle
    !kaggle competitions download -c ethz-cil-monocular-depth-estimation-2026 -p /content/
    !unzip -q /content/ethz-cil-monocular-depth-estimation-2026.zip -d {DATA_LOCAL}
    !rm /content/ethz-cil-monocular-depth-estimation-2026.zip
    print('Dataset ready.')
else:
    print('Dataset already on local SSD.')

if FORCE_CLEAN_LOCAL_SAM and os.path.exists(LOCAL_SAM_ROOT):
    shutil.rmtree(LOCAL_SAM_ROOT)
    print(f'Deleted local SAM root: {LOCAL_SAM_ROOT}')

Path(LOCAL_TRAIN_SAM_DIR).mkdir(parents=True, exist_ok=True)
Path(LOCAL_TEST_SAM_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_DATA_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_SAM_EXPORT).mkdir(parents=True, exist_ok=True)
Path(DRIVE_SAM_CHECKPOINTS).mkdir(parents=True, exist_ok=True)

from tools.colab_sam_preprocessing import SamPrepV2Config, ensure_sam_checkpoint

SAM_CHECKPOINT_PATH, SAM_DEVICE = ensure_sam_checkpoint(
    model_type=SAM_MODEL_TYPE,
    drive_dir=DRIVE_SAM_CHECKPOINTS,
)

sam_prep_cfg = SamPrepV2Config(
    data_root=DATASET_LOCAL,
    train_sam_dir=LOCAL_TRAIN_SAM_DIR,
    test_sam_dir=LOCAL_TEST_SAM_DIR,
    sam_checkpoint_path=str(SAM_CHECKPOINT_PATH),
    sam_model_type=SAM_MODEL_TYPE,
    seed=SEED,
    max_trainval_images=MAX_TRAINVAL_IMAGES,
    max_test_images=MAX_TEST_IMAGES,
    subset_policy=SUBSET_POLICY,
    part1_stats_csv=PART1_STATS_CSV,
    force_regenerate=FORCE_REGENERATE_SAM,
    points_per_side=POINTS_PER_SIDE,
    pred_iou_thresh=PRED_IOU_THRESH,
    stability_score_thresh=STABILITY_THRESH,
    mask_score_min=MASK_SCORE_MIN,
)

print(json.dumps(sam_prep_cfg.__dict__, indent=2))

Copying SAM checkpoint to local disk (/content/sam_vit_h_4b8939.pth) ...
SAM_CHECKPOINT_PATH: /content/sam_vit_h_4b8939.pth
SAM_MODEL_TYPE: vit_h | SAM device: cuda


#### Run SAM on entire dataset

In [ ]:
# run preprocessing
from tools.colab_sam_preprocessing import run_sam_preprocessing_workflow

SAM_RESULTS = run_sam_preprocessing_workflow(
    cfg=sam_prep_cfg,
    local_artifact_root=LOCAL_SAM_ROOT,
    sam_device=SAM_DEVICE,
    drive_export_root=DRIVE_SAM_EXPORT if SAVE_TO_DRIVE else None,
    train_zip_path=TRAIN_SAM_ZIP if SAVE_TO_DRIVE else None,
    test_zip_path=SAM_ZIP if SAVE_TO_DRIVE else None,
)

train_records = SAM_RESULTS['train_records']
test_records = SAM_RESULTS['test_records']

display(train_records.head())
display(test_records.head())

print(json.dumps(SAM_RESULTS['summary'], indent=2))
if SAM_RESULTS['drive_paths'] is not None:
    print(json.dumps(SAM_RESULTS['drive_paths'], indent=2))
if SAM_RESULTS['zip_paths'] is not None:
    print(json.dumps(SAM_RESULTS['zip_paths'], indent=2))

Train pool: 22605 | Train: 20345 | Val: 2260 | Test: 591


SAM v2 train:   0%|          | 0/20345 [00:00<?, ?it/s]

train: requested=20345 | created=20345 | skipped=0 | failed=0


SAM v2 val:   0%|          | 0/2260 [00:00<?, ?it/s]

val: requested=2260 | created=2260 | skipped=0 | failed=0


...

test: requested=591 | created=591 | skipped=0 | failed=0


,split,name,rgb_path,sam_npz_path,sam_npz_exists,depth_path,depth_exists
0,train,train_000000_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00000...,True,/content/data/monodepth_kaggle2026/train/train...,True
1,train,train_000001_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00000...,True,/content/data/monodepth_kaggle2026/train/train...,True
2,train,train_000002_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00000...,True,/content/data/monodepth_kaggle2026/train/train...,True
3,train,train_000003_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00000...,True,/content/data/monodepth_kaggle2026/train/train...,True
4,train,train_000004_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00000...,True,/content/data/monodepth_kaggle2026/train/train...,True


,split,name,rgb_path,sam_npz_path,sam_npz_exists,depth_path,depth_exists
0,val,train_000007_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00000...,True,/content/data/monodepth_kaggle2026/train/train...,True
1,val,train_000024_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00002...,True,/content/data/monodepth_kaggle2026/train/train...,True
2,val,train_000029_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00002...,True,/content/data/monodepth_kaggle2026/train/train...,True
3,val,train_000042_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00004...,True,/content/data/monodepth_kaggle2026/train/train...,True
4,val,train_000044_rgb.png,/content/data/monodepth_kaggle2026/train/train...,/content/cil_analysis_sam_v2/train/train_00004...,True,/content/data/monodepth_kaggle2026/train/train...,True


,split,name,rgb_path,sam_npz_path,sam_npz_exists
0,test,test_000000_rgb.png,/content/data/monodepth_kaggle2026/test/test_0...,/content/cil_analysis_sam_v2/test/test_000000_...,True
1,test,test_000001_rgb.png,/content/data/monodepth_kaggle2026/test/test_0...,/content/cil_analysis_sam_v2/test/test_000001_...,True
2,test,test_000002_rgb.png,/content/data/monodepth_kaggle2026/test/test_0...,/content/cil_analysis_sam_v2/test/test_000002_...,True
3,test,test_000003_rgb.png,/content/data/monodepth_kaggle2026/test/test_0...,/content/cil_analysis_sam_v2/test/test_000003_...,True
4,test,test_000004_rgb.png,/content/data/monodepth_kaggle2026/test/test_0...,/content/cil_analysis_sam_v2/test/test_000004_...,True


Saved SAM v2 outputs to: /content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622
{
  "train_records_csv": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622/train_records.csv",
  "val_records_csv": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622/val_records.csv",
  "test_records_csv": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622/test_records.csv",
  "train_sam_dir": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622/train_sam_v2",
  "test_sam_dir": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622/test_sam_v2",
  "config_json": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622/sam_prep_v2_config.json",
  "summary_json": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622/sam_prep_v2_summary.json"
}


#### Quick check

In [ ]:
# quick sanity check
from tools.colab_sam_preprocessing import inspect_sam_npz

counts = {
    'expected_train': SAM_RESULTS['summary']['expected_train'],
    'expected_test': SAM_RESULTS['summary']['expected_test'],
    'train_npz_count_on_disk': SAM_RESULTS['summary']['train_npz_count_on_disk'],
    'test_npz_count_on_disk': SAM_RESULTS['summary']['test_npz_count_on_disk'],
}
print(json.dumps(counts, indent=2))

sample_train = next(Path(LOCAL_TRAIN_SAM_DIR).glob('*_sam_v2.npz'), None)
sample_test = next(Path(LOCAL_TEST_SAM_DIR).glob('*_sam_v2.npz'), None)

if sample_train is not None:
    print(f'\nSample train NPZ: {sample_train.name}')
    print(json.dumps(inspect_sam_npz(sample_train), indent=2, default=str))

if sample_test is not None:
    print(f'\nSample test NPZ: {sample_test.name}')
    print(json.dumps(inspect_sam_npz(sample_test), indent=2, default=str))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_ROOT: /content/data/monodepth_kaggle2026
Loaded old config.
Train rows: 20345
Val rows:   2260
Test rows:  591
SAM model: vit_h
Checkpoint: /content/sam_vit_h_4b8939.pth
Device: cuda
[VERIFY] before copy phase | train_sam_v2=0 files | test_sam_v2=0 files


Copy existing partial NPZs:   0%|          | 0/23196 [00:00<?, ?it/s]

[VERIFY] copy progress 2000/23196 | train_sam_v2=438 files | test_sam_v2=0 files
[VERIFY] copy progress 4000/23196 | train_sam_v2=878 files | test_sam_v2=0 files
[VERIFY] copy progress 6000/23196 | train_sam_v2=1297 files | test_sam_v2=0 files
[VERIFY] copy progress 8000/23196 | train_sam_v2=1723 files | test_sam_v2=0 files
[VERIFY] copy progress 10000/23196 | train_sam_v2=2160 files | test_sam_v2=0 files
[VERIFY] copy progress 12000/23196 | train_sam_v2=2572 files | test_sam_v2=0 files
[VERIFY] copy progress 14000/23196 | train_sam_v2=2985 files | test_sam_v2=0 files
[VERIFY] copy progress 16000/23196 | train_sam_v2=3398 files | test_sam_v2=0 files
[VERIFY] copy progress 18000/23196 | train_sam_v2=3786 files | test_sam_v2=0 files
[VERIFY] copy progress 20000/23196 | train_sam_v2=4258 files | test_sam_v2=0 files
[VERIFY] copy progress 22000/23196 | train_sam_v2=4650 files | test_sam_v2=0 files
Copied existing NPZs from partial export: 4774
[VERIFY] after copy phase | train_sam_v2=4774 

Generate missing train:   0%|          | 0/16014 [00:00<?, ?it/s]

[VERIFY] generate train progress 2000/16014 | train_sam_v2=6774 files | test_sam_v2=0 files
[VERIFY] generate train progress 4000/16014 | train_sam_v2=8774 files | test_sam_v2=0 files
[VERIFY] generate train progress 6000/16014 | train_sam_v2=10774 files | test_sam_v2=0 files
[VERIFY] generate train progress 8000/16014 | train_sam_v2=12774 files | test_sam_v2=0 files
[VERIFY] generate train progress 10000/16014 | train_sam_v2=14774 files | test_sam_v2=0 files
[VERIFY] generate train progress 12000/16014 | train_sam_v2=16774 files | test_sam_v2=0 files
[VERIFY] generate train progress 14000/16014 | train_sam_v2=18774 files | test_sam_v2=0 files
[VERIFY] generate train progress 16000/16014 | train_sam_v2=20774 files | test_sam_v2=0 files
[VERIFY] after generate train | train_sam_v2=20788 files | test_sam_v2=0 files
[VERIFY] before generate val | train_sam_v2=20788 files | test_sam_v2=0 files


Generate missing val:   0%|          | 0/1817 [00:00<?, ?it/s]

[VERIFY] after generate val | train_sam_v2=22605 files | test_sam_v2=0 files
[VERIFY] before generate test | train_sam_v2=22605 files | test_sam_v2=0 files


Generate missing test:   0%|          | 0/591 [00:00<?, ?it/s]

[VERIFY] after generate test | train_sam_v2=22605 files | test_sam_v2=591 files

Verification
Expected train_sam_v2 count (train+val): 22605
Actual train_sam_v2 count:               22605
Expected test_sam_v2 count:              591
Actual test_sam_v2 count:                591

Saved recovered SAM v2 export to:
/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622_recovered
{
  "train_records_csv": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622_recovered/train_records.csv",
  "val_records_csv": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622_recovered/val_records.csv",
  "test_records_csv": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622_recovered/test_records.csv",
  "train_sam_dir": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622_recovered/train_sam_v2",
  "test_sam_dir": "/content/drive/MyDrive/CIL Project/sam_prep_v2_20260515_073622_recovered/test_sam_v2",
  "config_json": "/content/drive/MyDrive/CIL Project/

#### Downstream integration

In [ ]:
# downstream paths
SAM_LOCAL = LOCAL_SAM_ROOT

REUSABLE_SAM_PATHS = {
    'sam_local': SAM_LOCAL,
    'train_sam_dir': LOCAL_TRAIN_SAM_DIR,
    'test_sam_dir': LOCAL_TEST_SAM_DIR,
    'train_records_csv': SAM_RESULTS['local_paths']['train_records_csv'],
    'test_records_csv': SAM_RESULTS['local_paths']['test_records_csv'],
    'drive_export_root': DRIVE_SAM_EXPORT if SAVE_TO_DRIVE else None,
    'train_sam_zip': TRAIN_SAM_ZIP if SAVE_TO_DRIVE else None,
    'test_sam_zip': SAM_ZIP if SAVE_TO_DRIVE else None,
}

print(json.dumps(REUSABLE_SAM_PATHS, indent=2))
print('\nFor downstream refinement in colab_train.ipynb:')
print(f"SAM_LOCAL = '{SAM_LOCAL}'")
print(f"train_refiner.py --sam_dir {LOCAL_TRAIN_SAM_DIR}")
print(f"predict_refiner.py --test_sam_dir {LOCAL_TEST_SAM_DIR}")

{
  "sam_prep_dir_exists": true,
  "train_sam_dir_exists": true,
  "test_sam_dir_exists": true,
  "train_npz_count_on_disk": 22605,
  "test_npz_count_on_disk": 591,
  "expected_train_plus_val": 22605,
  "expected_test": 591,
  "checks": {
    "train": {
      "split": "train",
      "expected_rows": 20345,
      "found_files_for_rows": 20345,
      "missing_count": 0,
      "first_missing_examples": []
    },
    "val": {
      "split": "val",
      "expected_rows": 2260,
      "found_files_for_rows": 2260,
      "missing_count": 0,
      "first_missing_examples": []
    },
    "test": {
      "split": "test",
      "expected_rows": 591,
      "found_files_for_rows": 591,
      "missing_count": 0,
      "first_missing_examples": []
    }
  }
}

OVERALL STATUS: PASS

Sample train NPZ: train_010441_rgb_sam_v2.npz
{
  "label_top1": [
    560,
    560
  ],
  "label_top2": [
    560,
    560
  ],
  "conf_top1": [
    560,
    560
  ],
  "conf_top2": [
    560,
    560
  ],
  "pixel_uncertai